# 04c — Temporal / Prospective Prediction

**Input:** `../data/raw/ema_personality_plus_surveys_merged.csv`
**Output:** `../outputs/tables/temporal_prediction_summary.csv`

**Description:**

Four temporal designs testing whether text adds value beyond numeric ratings for predicting **future** crisis states:

1. **PM → next-day PM crisis** (continuous Ridge) — does today's text predict tomorrow?
2. **AM → same-day PM crisis** (continuous Ridge) — does morning predict afternoon?
3. **AM + other AM ratings → same-day PM** (continuous Ridge) — full AM baseline
4. **PM + other PM ratings → next-morning AM** (continuous Ridge) — evening to next morning

Each design: baseline (numeric only) vs full (numeric + text PCA), GroupKFold CV.

Note: This notebook loads raw data and builds its own day-level tables because the temporal designs require lagging and AM/PM alignment not present in the standard pipeline.

Matches original cells 6–10.

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "raw", "ema_personality_plus_surveys_merged.csv")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
DT_COL = "ema_dt"
TEXT_COL = "ema_text"

AM_TOTAL = "dailyAM__Total Score from 5 Questions"
PM_TOTAL = "dailyPM__Total Score from 5 Questions"

PM_START_HOUR = 15
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
N_SPLITS = 5
N_PCS = 20
RIDGE_ALPHA = 10.0
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================
# HELPERS
# =========================
def first_nonnull(x):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan


def find_crisis_item_cols(df, prefix, total_col, top_k=5):
    cols = [c for c in df.columns if c.startswith(prefix) and pd.api.types.is_numeric_dtype(df[c])]
    candidates = []
    for c in cols:
        if c == total_col:
            continue
        s = df[c].dropna()
        if len(s) < 50:
            continue
        mn, mx = s.min(), s.max()
        if mn >= 0 and mx <= 4:
            frac_nonint = np.mean(np.abs(s - np.round(s)) > 1e-6)
            if frac_nonint < 0.01 and s.nunique() == 5:
                tmp = df[[c, total_col]].dropna()
                if len(tmp) < 100:
                    continue
                corr = tmp.corr().iloc[0, 1]
                candidates.append((c, float(corr)))
    candidates = sorted(candidates, key=lambda x: -abs(x[1]))
    return [c for c, _ in candidates[:top_k]]


def grouped_cv_ridge(X, y, groups, label=""):
    gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(groups))))
    maes, rmses, r2s = [], [], []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED))
        ])
        pipe.fit(X[tr], y[tr])
        yhat = pipe.predict(X[te])
        maes.append(mean_absolute_error(y[te], yhat))
        rmses.append(np.sqrt(mean_squared_error(y[te], yhat)))
        r2s.append(r2_score(y[te], yhat))
    return {
        "MAE_mean": float(np.mean(maes)), "MAE_sd": float(np.std(maes)),
        "RMSE_mean": float(np.mean(rmses)), "RMSE_sd": float(np.std(rmses)),
        "R2_mean": float(np.mean(r2s)), "R2_sd": float(np.std(r2s)),
    }


def grouped_cv_text_plus_numeric(X_text, X_num, y, groups, label=""):
    gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(groups))))
    maes, rmses, r2s = [], [], []
    for fold, (tr, te) in enumerate(gkf.split(X_text, y, groups=groups), start=1):
        pca = PCA(n_components=min(N_PCS, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])
        Xtr = np.hstack([X_num[tr], Ttr])
        Xte = np.hstack([X_num[te], Tte])
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED))
        ])
        pipe.fit(Xtr, y[tr])
        yhat = pipe.predict(Xte)
        maes.append(mean_absolute_error(y[te], yhat))
        rmses.append(np.sqrt(mean_squared_error(y[te], yhat)))
        r2s.append(r2_score(y[te], yhat))
    return {
        "MAE_mean": float(np.mean(maes)), "MAE_sd": float(np.std(maes)),
        "RMSE_mean": float(np.mean(rmses)), "RMSE_sd": float(np.std(rmses)),
        "R2_mean": float(np.mean(r2s)), "R2_sd": float(np.std(r2s)),
    }


def get_or_compute_embeddings(texts, cache_path, model):
    if os.path.exists(cache_path):
        X = np.load(cache_path)
        if X.shape[0] == len(texts):
            print(f"  Loaded cached embeddings: {X.shape}")
            return X
        os.remove(cache_path)
    X = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache_path, X)
    print(f"  Computed + saved embeddings: {X.shape}")
    return X

In [ ]:
# =========================
# LOAD RAW DATA
# =========================
df = pd.read_csv(DATA_PATH)

if PID_COL not in df.columns:
    PID_COL = "expiwell_id"
if DT_COL not in df.columns:
    DT_COL = "start_date"

df[DT_COL] = pd.to_datetime(df[DT_COL], errors="coerce")
df["ema_date"] = df[DT_COL].dt.date
df["hour"] = df[DT_COL].dt.hour

df["crisis_AM"] = df[AM_TOTAL]
df["crisis_PM"] = df[PM_TOTAL]

# Identify "other" AM and PM rating columns (excluding crisis items)
CRISIS_AM_COLS = find_crisis_item_cols(df, "dailyAM__", AM_TOTAL, top_k=5)
CRISIS_PM_COLS = find_crisis_item_cols(df, "dailyPM__", PM_TOTAL, top_k=5)

OTHER_AM_COLS = [
    c for c in df.columns
    if c.startswith("dailyAM__") and pd.api.types.is_numeric_dtype(df[c])
    and c not in set([AM_TOTAL] + CRISIS_AM_COLS)
]

OTHER_PM_COLS = [
    c for c in df.columns
    if c.startswith("dailyPM__") and pd.api.types.is_numeric_dtype(df[c])
    and c not in set([PM_TOTAL] + CRISIS_PM_COLS)
]

print("Loaded rows:", len(df), "| participants:", df[PID_COL].nunique())
print("Other AM covariates:", len(OTHER_AM_COLS))
print("Other PM covariates:", len(OTHER_PM_COLS))

st_model = SentenceTransformer(EMBED_MODEL)

In [ ]:
# =========================
# BUILD DAY TABLE (all text, AM+PM outcomes)
# =========================
agg_dict = {
    "day_text": (TEXT_COL, lambda s: " ".join([str(t) for t in s.dropna()])),
    "n_text": (TEXT_COL, lambda s: int(s.notna().sum())),
    "crisis_AM": ("crisis_AM", first_nonnull),
    "crisis_PM": ("crisis_PM", first_nonnull),
}
for c in OTHER_AM_COLS:
    agg_dict[c] = (c, first_nonnull)
for c in OTHER_PM_COLS:
    agg_dict[c] = (c, first_nonnull)

day = (
    df.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(**agg_dict)
)

day["day_text"] = day["day_text"].fillna("").astype(str)
day["has_text"] = day["day_text"].str.strip().ne("")
day = day.loc[day["has_text"]].reset_index(drop=True)

# Also build PM-window text
pm_text_df = df.loc[df["hour"].notna() & (df["hour"] >= PM_START_HOUR)].copy()
pm_text_day = (
    pm_text_df.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(
        pm_text=(TEXT_COL, lambda s: " ".join([str(t) for t in s.dropna()])),
        n_pm_text=(TEXT_COL, lambda s: int(s.notna().sum()))
    )
)
day = day.merge(pm_text_day, on=[PID_COL, "ema_date"], how="left")
day["pm_text"] = day["pm_text"].fillna("").astype(str)

# Sort for lagging
day = day.sort_values([PID_COL, "ema_date"]).reset_index(drop=True)
day["crisis_PM_next"] = day.groupby(PID_COL)["crisis_PM"].shift(-1)
day["crisis_AM_next"] = day.groupby(PID_COL)["crisis_AM"].shift(-1)

print("Day rows:", len(day), "| participants:", day[PID_COL].nunique())

In [ ]:
# =========================
# HELPER: run one temporal design
# =========================
CACHE_DIR = os.path.join("..", "data", "processed")
all_results = []


def run_temporal_design(design_name, day_sub, text_col, y_col, baseline_cols, embed_tag):
    print(f"\n{'='*60}")
    print(f"Design: {design_name}")
    print(f"{'='*60}")

    # Filter to usable rows
    day_sub = day_sub.copy()
    mask = day_sub[y_col].notna() & day_sub[text_col].str.strip().ne("")
    for bc in baseline_cols:
        mask &= day_sub[bc].notna()
    day_m = day_sub.loc[mask].reset_index(drop=True)

    if len(day_m) < 50:
        print(f"  Skipping: only {len(day_m)} usable rows")
        return

    y = day_m[y_col].values.astype(float)
    groups = day_m[PID_COL].astype(str).values

    # Build numeric baseline
    X_num_df = day_m[baseline_cols].copy()
    X_num_df = X_num_df.dropna(axis=1, how="all")
    X_num = X_num_df.fillna(X_num_df.mean()).values.astype(float)
    nan_cols = np.any(np.isnan(X_num), axis=0)
    if nan_cols.any():
        X_num = X_num[:, ~nan_cols]

    # Embeddings
    cache_path = os.path.join(CACHE_DIR, f"emb_temporal_{embed_tag}.npy")
    X_text = get_or_compute_embeddings(day_m[text_col].tolist(), cache_path, st_model)

    print(f"  N={len(day_m)} | participants={day_m[PID_COL].nunique()}")
    print(f"  y mean={y.mean():.3f} sd={y.std():.3f}")
    print(f"  Baseline features: {X_num.shape[1]} | Text dims: {X_text.shape[1]}")

    # Baseline
    res_base = grouped_cv_ridge(X_num, y, groups, label="Baseline")
    print(f"  Baseline: MAE={res_base['MAE_mean']:.3f}±{res_base['MAE_sd']:.3f} R²={res_base['R2_mean']:.3f}±{res_base['R2_sd']:.3f}")

    # Full (+text)
    res_full = grouped_cv_text_plus_numeric(X_text, X_num, y, groups, label="Full")
    print(f"  Full:     MAE={res_full['MAE_mean']:.3f}±{res_full['MAE_sd']:.3f} R²={res_full['R2_mean']:.3f}±{res_full['R2_sd']:.3f}")

    delta_r2 = res_full["R2_mean"] - res_base["R2_mean"]
    print(f"  ΔR² = {delta_r2:.4f}")

    all_results.append({"design": design_name, "model": "baseline", "n": len(day_m), **res_base})
    all_results.append({"design": design_name, "model": "full_text", "n": len(day_m), **res_full})
    all_results.append({"design": design_name, "model": "delta", "n": len(day_m),
                        "Delta_MAE": res_base["MAE_mean"] - res_full["MAE_mean"],
                        "Delta_RMSE": res_base["RMSE_mean"] - res_full["RMSE_mean"],
                        "Delta_R2": delta_r2})

In [ ]:
# =========================
# DESIGN 1: PM crisis → next-day PM crisis
# =========================
run_temporal_design(
    "PM_to_nextPM",
    day,
    text_col="day_text",
    y_col="crisis_PM_next",
    baseline_cols=["crisis_PM"],
    embed_tag="pm_to_nextpm"
)

In [ ]:
# =========================
# DESIGN 2: AM crisis → same-day PM crisis
# =========================
run_temporal_design(
    "AM_to_samePM",
    day,
    text_col="day_text",
    y_col="crisis_PM",
    baseline_cols=["crisis_AM"],
    embed_tag="am_to_samepm"
)

In [ ]:
# =========================
# DESIGN 3: AM + other AM ratings → same-day PM
# =========================
run_temporal_design(
    "AM_full_to_samePM",
    day,
    text_col="day_text",
    y_col="crisis_PM",
    baseline_cols=["crisis_AM"] + OTHER_AM_COLS,
    embed_tag="am_full_to_samepm"
)

In [ ]:
# =========================
# DESIGN 4: PM + other PM ratings → next-morning AM
# =========================
run_temporal_design(
    "PM_full_to_nextAM",
    day,
    text_col="pm_text",
    y_col="crisis_AM_next",
    baseline_cols=["crisis_PM"] + OTHER_PM_COLS,
    embed_tag="pm_full_to_nextam"
)

In [ ]:
# =========================
# SAVE SUMMARY
# =========================
summary = pd.DataFrame(all_results)
summary_path = os.path.join(OUT_DIR, "temporal_prediction_summary.csv")
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print("TEMPORAL PREDICTION SUMMARY")
print("=" * 60)
print(summary.to_string(index=False))
print("\nSaved:", summary_path)